In [0]:
%pip install -U -qqqq mlflow databricks-openai databricks-agents databricks-langchain
dbutils.library.restartPython()

In [0]:
from langchain.agents import create_agent
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient
import mlflow
from databricks_langchain import ChatDatabricks


### Using Python function as a tool

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
import uuid

# Initialize Spark Session

@tool
def split_sql_into_sections(sql_content):
    """
    Split SQL content into logically complete sections
    """
    sections = []
    
    # Remove comments and normalize whitespace
    sql_content = re.sub(r'--.*?\n', '\n', sql_content)
    sql_content = re.sub(r'/\*.*?\*/', '', sql_content, flags=re.DOTALL)
    sql_content = re.sub(r'\s+', ' ', sql_content).strip()
    
    if not sql_content:
        return sections
    
    # Split by semicolons first
    potential_sections = re.split(r';\s*', sql_content)
    
    for section in potential_sections:
        section = section.strip()
        if not section:
            continue
            
        # Further split large sections by logical boundaries
        if len(section) > 2000:  # If section is too large
            # Split by CTE boundaries
            cte_pattern = r'(?<=\))\s*,\s*(?=\w+\s+AS\s*\()'
            cte_splits = re.split(cte_pattern, section)
            
            for cte_section in cte_splits:
                cte_section = cte_section.strip()
                if cte_section:
                    # Further split by subquery boundaries if still too large
                    if len(cte_section) > 1500:
                        subquery_pattern = r'(?<=\))\s+(?=(?:UNION|INTERSECT|EXCEPT|ORDER BY|GROUP BY))'
                        sub_splits = re.split(subquery_pattern, cte_section, flags=re.IGNORECASE)
                        sections.extend([s.strip() for s in sub_splits if s.strip()])
                    else:
                        sections.append(cte_section)
        else:
            sections.append(section)
    
    return sections


In [0]:
chat_model = ChatDatabricks(
    endpoint="databricks-claude-3-7-sonnet",
    temperature=0.1,
    max_tokens=256,
)

In [0]:
builtin_tools=[split_sql_into_sections]
agent = create_agent(
    tools=builtin_tools,
    model=chat_model
)
agent.invoke({"messages":"Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"})

### Failed tool usage attempts

Python function requiring external library

In [0]:

from langchain.tools import tool
@tool
def split_sql(sql_content):
    """
    Split SQL content into logically complete sections
    """
    
    sections=sqlparse.split(sql_content)
    # sqlparse.format(sql_content, reindent=True, keyword_case='upper')
    return sections 

In [0]:
builtin_tools=[split_sql]
agent = create_agent(
    tools=builtin_tools,
    model=chat_model
)
agent.invoke({"messages":"Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"})

Python UDF registered on UC

In [0]:
%sql
USE CATALOG learn_adb_fikrat;
USE bronze;
CREATE OR REPLACE FUNCTION sql_split_custom(sql_content STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Returns individual SQL commands from whole SQL script'
AS
$$
sections = []

# Remove comments and normalize whitespace
sql_content = re.sub(r'--.*?\n', '\n', sql_content)
sql_content = re.sub(r'/\*.*?\*/', '', sql_content, flags=re.DOTALL)
sql_content = re.sub(r'\s+', ' ', sql_content).strip()

if not sql_content:
    return sections

# Split by semicolons first
potential_sections = re.split(r';\s*', sql_content)

for section in potential_sections:
    section = section.strip()
    if not section:
        continue
        
    # Further split large sections by logical boundaries
    if len(section) > 2000:  # If section is too large
        # Split by CTE boundaries
        cte_pattern = r'(?<=\))\s*,\s*(?=\w+\s+AS\s*\()'
        cte_splits = re.split(cte_pattern, section)
        
        for cte_section in cte_splits:
            cte_section = cte_section.strip()
            if cte_section:
                # Further split by subquery boundaries if still too large
                if len(cte_section) > 1500:
                    subquery_pattern = r'(?<=\))\s+(?=(?:UNION|INTERSECT|EXCEPT|ORDER BY|GROUP BY))'
                    sub_splits = re.split(subquery_pattern, cte_section, flags=re.IGNORECASE)
                    sections.extend([s.strip() for s in sub_splits if s.strip()])
                else:
                    sections.append(cte_section)
    else:
        sections.append(section)

return sections
$$;

In [0]:
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(function_names=["learn_adb_fikrat.bronze.sql_split_custom"], client=client).tools

In [0]:
agent = create_agent(
    tools=builtin_tools,
    model=chat_model
)
agent.invoke({"messages":"Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"})

In [0]:
%sql
USE CATALOG learn_adb_fikrat;
USE bronze;
CREATE OR REPLACE FUNCTION sql_split(sql_content STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Returns individual SQL commands from whole SQL script'
AS
$$
import sqlparse 
sections=sqlparse.split(sql_content)
return sections 
$$;

In [0]:
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(function_names=["learn_adb_fikrat.bronze.sql_split"], client=client).tools

In [0]:
agent = create_agent(
    tools=builtin_tools,
    model=chat_model
)
agent.invoke({"messages":"Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"})

In [0]:
# from langchain_core.prompts import ChatPromptTemplate
# my_prompt = ChatPromptTemplate.from_messages(
#     [
#         ("user", "Can you split this command into sections: {sql_command}"),
#     ]
# )

# my_prompt2="Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"
# chain=prompt|chat_model
agent = create_agent(
    tools=builtin_tools,
    model=chat_model
    # system_prompt=my_prompt2
    # verbose=True,
)
# agent.invoke({"input":"Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"})
agent.invoke(
    "Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"
)

In [0]:
%sql
DROP TABLE IF EXISTS learn_adb_fikrat.cc_bronze.source_target_queries;
DROP TABLE IF EXISTS learn_adb_fikrat.cc_bronze.source_target_commands; 
CREATE TABLE learn_adb_fikrat.cc_bronze.source_target_queries
(
  query_id STRING,
  source_query STRING,
  source_file STRING,
  target_query STRING,
  total_commands INT
);

CREATE TABLE learn_adb_fikrat.cc_bronze.source_target_commands (
  query_id STRING,
  command_number INT,
  source_command STRING,
  command_type STRING,
  target_command STRING
   )

In [0]:
sql_content="Select * from Bronze.test; Select * from Bronze.test2"
# print(split_sql(sql_content))
# print(split_sql_into_sections(sql_content))

In [0]:
Convert this UDF to Table UDF returning multiple rows


In [0]:
%sql
USE CATALOG learn_adb_fikrat;
USE bronze;
CREATE OR REPLACE FUNCTION sql_split(sql_content STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Returns individual SQL commands from whole SQL script'
AS
$$
import sqlparse 
sections=sqlparse.split(sql_content)
return sections 
$$;

In [0]:
spark.sql(f"select sql_split('{sql_content}')")

In [0]:
agent.invoke({"input":"Split this command into sections: 'Select * from Bronze.test; Select * from Bronze.test2'"})

In [0]:
%sql
SELECT `learn_adb_fikrat`.`bronze`.`sql_split`("Select * from Bronze.test; Select * from Bronze.test2")

In [0]:
import mlflow
prompt = mlflow.genai.register_prompt(
    name="learn_adb_fikrat.bronze.sql_split",
    template="You are a helpful assistant. Split this command: {{sql_command}}",
    commit_message="Initial customer support prompt"
)

In [0]:
mlflow.genai.set_prompt_alias(
    name="learn_adb_fikrat.bronze.sql_split",
    alias="dev",
    version=1
)

In [0]:
prompt = mlflow.genai.load_prompt(name_or_uri="prompts:/learn_adb_fikrat.bronze.sql_split@dev")
response = agent.invoke(prompt.format(sql_command=sql_content))

In [0]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a chatbot that can help splitting SQL commands into sections {sql_command}.",
        ),
        ("user", "{question}"),
    ]
)
prompt.format_messages(question="Split this command into sections", sql_command=sql_content)
print (prompt)
# agent.invoke( prompt)

In [0]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("user", "Can you split this command into sections: {sql_command}"),
    ]
)
# prompt.format_messages(question="Split this command into sections {sql_command}")
chain=prompt| agent
chain.invoke(
    {
        "sql_command": 'Select * from Bronze.test; Select * from Bronze.test2'
    })

In [0]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a chatbot that can help splitting SQL commands into sections {sql_command}.",
        ),
        ("user", "{question}"),
    ]
)
# prompt.format_messages(question="Split this command into sections {sql_command}")
# chain=prompt| chat_model
agent.invoke(
    {
        "question": "Can you split this command into sections","sql_command": 'Select * from Bronze.test; Select * from Bronze.test2'
    })

In [0]:

# 1. Define the function to modify a UC table
@tool
def python_udf_summarize(x, y):
    """Returns summary of two numeric parameters"""
    return x + y

# Another tool example
@tool
def python_udf_multiple(x, y):
    """Returns multiplication of two numeric parameters"""
    return x * y


tools=[python_udf_summarize,python_udf_multiple]
# 3. Add the tool to your agent
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4", temperature=0.1)
agent = create_agent(
    tools=tools,
    model=llm
)

# Example usage
agent.invoke({"messages": "Summarize 5 and 7 multplied to 10"})

In [0]:

def convert_mssql_to_spark_sql(sql_query):
    """
    Convert MS SQL syntax to Spark SQL syntax
    """
    converted_query = sql_query
    
    # Data type conversions
    data_type_conversions = {
        r'\bNVARCHAR\s*\(\s*(\d+)\s*\)': r'STRING',
        r'\bVARCHAR\s*\(\s*(\d+)\s*\)': r'STRING',
        r'\bNVARCHAR\s*\(\s*MAX\s*\)': r'STRING',
        r'\bVARCHAR\s*\(\s*MAX\s*\)': r'STRING',
        r'\bTEXT\b': 'STRING',
        r'\bNTEXT\b': 'STRING',
        r'\bCHAR\s*\(\s*(\d+)\s*\)': r'STRING',
        r'\bNCHAR\s*\(\s*(\d+)\s*\)': r'STRING',
        r'\bBIT\b': 'BOOLEAN',
        r'\bTINYINT\b': 'TINYINT',
        r'\bSMALLINT\b': 'SMALLINT',
        r'\bINT\b': 'INT',
        r'\bBIGINT\b': 'BIGINT',
        r'\bFLOAT\b': 'DOUBLE',
        r'\bREAL\b': 'FLOAT',
        r'\bDATETIME\b': 'TIMESTAMP',
        r'\bDATETIME2\b': 'TIMESTAMP',
        r'\bSMALLDATETIME\b': 'TIMESTAMP',
        r'\bDATE\b': 'DATE',
        r'\bTIME\b': 'STRING',
        r'\bMONEY\b': 'DECIMAL(19,4)',
        r'\bSMALLMONEY\b': 'DECIMAL(10,4)',
        r'\bDECIMAL\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)': r'DECIMAL(\1,\2)',
        r'\bNUMERIC\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)': r'DECIMAL(\1,\2)',
        r'\bUNIQUEIDENTIFIER\b': 'STRING'
    }
    
    for pattern, replacement in data_type_conversions.items():
        converted_query = re.sub(pattern, replacement, converted_query, flags=re.IGNORECASE)
    
    # Function conversions
    function_conversions = {
        # Null handling
        r'\bISNULL\s*\(\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'COALESCE(\1, \2)',
        
        # Date functions
        r'\bGETDATE\s*\(\s*\)': 'CURRENT_TIMESTAMP()',
        r'\bGETUTCDATE\s*\(\s*\)': 'CURRENT_TIMESTAMP()',
        r'\bSYSDATETIME\s*\(\s*\)': 'CURRENT_TIMESTAMP()',
        r'\bCURRENT_TIMESTAMP\b': 'CURRENT_TIMESTAMP()',
        
        # String functions
        r'\bLEN\s*\(': 'LENGTH(',
        r'\bCHARINDEX\s*\(\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'LOCATE(\1, \2)',
        r'\bCHARINDEX\s*\(\s*([^,]+)\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'LOCATE(\1, \2, \3)',
        r'\bSTUFF\s*\(\s*([^,]+)\s*,\s*([^,]+)\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'INSERT(\1, \2, \3, \4)',
        r'\bLEFT\s*\(\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'SUBSTRING(\1, 1, \2)',
        r'\bRIGHT\s*\(\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'RIGHT(\1, \2)',
        r'\bREPLICATE\s*\(': 'REPEAT(',
        
        # Date arithmetic
        r'\bDATEDIFF\s*\(\s*DAY\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'DATEDIFF(\2, \1)',
        r'\bDATEDIFF\s*\(\s*MONTH\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'MONTHS_BETWEEN(\2, \1)',
        r'\bDATEDIFF\s*\(\s*YEAR\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'YEAR(\2) - YEAR(\1)',
        r'\bDATEADD\s*\(\s*DAY\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'DATE_ADD(\2, \1)',
        r'\bDATEADD\s*\(\s*MONTH\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'ADD_MONTHS(\2, \1)',
        r'\bDATEADD\s*\(\s*YEAR\s*,\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'ADD_MONTHS(\2, \1 * 12)',
        
        # Date parts
        r'\bDATEPART\s*\(\s*YEAR\s*,\s*([^)]+)\s*\)': r'YEAR(\1)',
        r'\bDATEPART\s*\(\s*MONTH\s*,\s*([^)]+)\s*\)': r'MONTH(\1)',
        r'\bDATEPART\s*\(\s*DAY\s*,\s*([^)]+)\s*\)': r'DAY(\1)',
        r'\bDATEPART\s*\(\s*HOUR\s*,\s*([^)]+)\s*\)': r'HOUR(\1)',
        r'\bDATEPART\s*\(\s*MINUTE\s*,\s*([^)]+)\s*\)': r'MINUTE(\1)',
        r'\bDATEPART\s*\(\s*SECOND\s*,\s*([^)]+)\s*\)': r'SECOND(\1)',
        
        # Conversion functions
        r'\bCAST\s*\(\s*([^)]+)\s+AS\s+([^)]+)\s*\)': r'CAST(\1 AS \2)',
        r'\bCONVERT\s*\(\s*([^,]+)\s*,\s*([^)]+)\s*\)': r'CAST(\2 AS \1)',
        
        # Mathematical functions
        r'\bCEILING\s*\(': 'CEIL(',
        r'\bPOWER\s*\(': 'POW(',
        
        # Aggregate functions
        r'\bSTDEV\s*\(': 'STDDEV(',
        r'\bSTDEVP\s*\(': 'STDDEV_POP(',
        r'\bVAR\s*\(': 'VARIANCE(',
        r'\bVARP\s*\(': 'VAR_POP(',
    }
    
    for pattern, replacement in function_conversions.items():
        converted_query = re.sub(pattern, replacement, converted_query, flags=re.IGNORECASE)
    
    # Syntax conversions
    syntax_conversions = {
        # TOP clause
        r'\bSELECT\s+TOP\s+(\d+)\s+': r'SELECT ',
        r'\bSELECT\s+TOP\s+\(\s*(\d+)\s*\)\s+': r'SELECT ',
        
        # Square brackets to backticks
        r'\[([^\]]+)\]': r'`\1`',
        
        # Table hints removal
        r'\bWITH\s*\(\s*NOLOCK\s*\)': '',
        r'\bWITH\s*\(\s*UPDLOCK\s*\)': '',
        r'\bWITH\s*\(\s*READPAST\s*\)': '',
        r'\bWITH\s*\(\s*ROWLOCK\s*\)': '',
        r'\bWITH\s*\(\s*TABLOCK\s*\)': '',
        
        # IDENTITY columns
        r'\bIDENTITY\s*\(\s*\d+\s*,\s*\d+\s*\)': '',
        
        # Boolean expressions
        r'\bCASE\s+WHEN\s+(.+?)\s+THEN\s+1\s+ELSE\s+0\s+END\b': r'CASE WHEN \1 THEN TRUE ELSE FALSE END',
        
        # String concatenation
        r"(\w+)\s*\+\s*'([^']*)'": r"CONCAT(\1, '\2')",
        r"'([^']*)'\s*\+\s*(\w+)": r"CONCAT('\1', \2)",
        
        # ISNUMERIC function
        r'\bISNUMERIC\s*\(\s*([^)]+)\s*\)': r"(\1 RLIKE '^[0-9]+\.?[0-9]*$')",
        
        # String comparison
        r'\bCOLLATE\s+\w+': '',
    }
    
    for pattern, replacement in syntax_conversions.items():
        converted_query = re.sub(pattern, replacement, converted_query, flags=re.IGNORECASE)
    
    # Handle LIMIT clause for TOP
    top_match = re.search(r'\bSELECT\s+TOP\s+(\d+)\s+', sql_query, re.IGNORECASE)
    if top_match:
        limit_value = top_match.group(1)
        if not re.search(r'\bLIMIT\s+\d+', converted_query, re.IGNORECASE):
            converted_query += f' LIMIT {limit_value}'
    
    # Clean up extra whitespace
    converted_query = re.sub(r'\s+', ' ', converted_query).strip()
    
    return converted_query

def assemble_converted_sections(sections):
    """
    Assemble all converted query sections into a single query
    """
    if not sections:
        return ""
    
    # Join sections with semicolons and proper spacing
    assembled_query = ""
    for i, section in enumerate(sections):
        if section.strip():
            if i > 0:
                assembled_query += ";\n\n"
            assembled_query += section.strip()
    
    return assembled_query

def process_source_queries():
    """
    Main function to process source queries through all steps
    """
    try:
        print("Step 1: Reading source queries...")
        
        # Read from source table
        source_df = spark.sql("""
            SELECT 
                file_name,
                file_path,
                query_content as original_content,
                file_id
            FROM learn_adb_fikrat.cc_bronze.source_queries
            WHERE query_content IS NOT NULL AND query_content != ''
        """)
        
        source_data = source_df.collect()
        processed_records = []
        
        print(f"Found {len(source_data)} files to process")
        
        for row in source_data:
            file_name = row.get('file_name', 'unknown')
            file_path = row.get('file_path', '')
            original_query = row.get('original_content', '')
            file_id = row.get('file_id', str(uuid.uuid4()))
            
            if not original_query.strip():
                continue
                
            print(f"Processing file: {file_name}")
            
            # Step 2: Split content into sections
            sections = split_sql_into_sections(original_query)
            print(f"  Split into {len(sections)} sections")
            
            # Step 4: Convert each section to Spark SQL
            converted_sections = []
            
            for i, section in enumerate(sections):
                if section.strip():
                    converted_section = convert_mssql_to_spark_sql(section)
                    converted_sections.append(converted_section)
                    
                    # Step 3: Prepare record for each section
                    processed_records.append({
                        'file_id': file_id,
                        'file_name': file_name,
                        'file_path': file_path,
                        'section_number': i + 1,
                        'query': original_query,  # Original complete query
                        'query_section': section,  # Original section
                        'converted_query_section': converted_section,  # Converted section
                        'converted_query': '',  # Will be filled later
                        'processed_timestamp': current_timestamp()
                    })
            
            # Step 5: Assemble all converted sections
            assembled_query = assemble_converted_sections(converted_sections)
            
            # Update all records for this file with the assembled query
            for record in processed_records:
                if record['file_id'] == file_id:
                    record['converted_query'] = assembled_query
        
        # Create DataFrame and write to table
        if processed_records:
            print(f"Writing {len(processed_records)} processed sections to table...")
            
            schema = StructType([
                StructField("file_id", StringType(), True),
                StructField("file_name", StringType(), True),
                StructField("file_path", StringType(), True),
                StructField("section_number", IntegerType(), True),
                StructField("query", StringType(), True),
                StructField("query_section", StringType(), True),
                StructField("converted_query_section", StringType(), True),
                StructField("converted_query", StringType(), True),
                StructField("processed_timestamp", TimestampType(), True)
            ])
            
            processed_df = spark.createDataFrame(processed_records, schema)
            
            # Write to table
            processed_df.write \
                .mode("overwrite") \
                .option("mergeSchema", "true") \
                .saveAsTable("learn_adb_fikrat.cc_bronze.source_queries")
            
            print("Successfully completed all processing steps!")
            
            # Show summary
            summary_df = processed_df.groupBy("file_name") \
                .agg(
                    count("section_number").alias("total_sections"),
                    max("section_number").alias("max_section_number")
                )
            
            print("Processing Summary:")
            summary_df.show(truncate=False)
            
            # Show sample results
            print("\nSample converted sections:")
            processed_df.select(
                "file_name", 
                "section_number", 
                "query_section", 
                "converted_query_section"
            ).show(3, truncate=False)
            
        else:
            print("No records to process")
            
    except Exception as e:
        print(f"Error processing source queries: {str(e)}")
        raise e



Splitter function

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
import uuid

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("MS SQL to Spark SQL Converter") \
    .enableHiveSupport() \
    .getOrCreate()

def split_sql_into_sections(sql_content):
    """
    Split SQL content into logically complete sections
    """
    sections = []
    
    # Remove comments and normalize whitespace
    sql_content = re.sub(r'--.*?\n', '\n', sql_content)
    sql_content = re.sub(r'/\*.*?\*/', '', sql_content, flags=re.DOTALL)
    sql_content = re.sub(r'\s+', ' ', sql_content).strip()
    
    if not sql_content:
        return sections
    
    # Split by semicolons first
    potential_sections = re.split(r';\s*', sql_content)
    
    for section in potential_sections:
        section = section.strip()
        if not section:
            continue
            
        # Further split large sections by logical boundaries
        if len(section) > 2000:  # If section is too large
            # Split by CTE boundaries
            cte_pattern = r'(?<=\))\s*,\s*(?=\w+\s+AS\s*\()'
            cte_splits = re.split(cte_pattern, section)
            
            for cte_section in cte_splits:
                cte_section = cte_section.strip()
                if cte_section:
                    # Further split by subquery boundaries if still too large
                    if len(cte_section) > 1500:
                        subquery_pattern = r'(?<=\))\s+(?=(?:UNION|INTERSECT|EXCEPT|ORDER BY|GROUP BY))'
                        sub_splits = re.split(subquery_pattern, cte_section, flags=re.IGNORECASE)
                        sections.extend([s.strip() for s in sub_splits if s.strip()])
                    else:
                        sections.append(cte_section)
        else:
            sections.append(section)
    
    return sections

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import sqlparse,os,uuid
from sqlparse.sql import Statement
from sqlparse.tokens import Keyword, DML

def split_sql_with_sqlparse(sql_content):
    """
    Split SQL query into individual commands using sqlparse library.
    Returns a list of parsed SQL statements.
    """
    try:
        # Parse the SQL content
        # split_on_semicolon=True will split multiple statements
        parsed_statements = sqlparse.parse(sql_content)
        
        # Extract and clean each statement
        sections = []
        for statement in parsed_statements:
            # Format the statement for better readability
            formatted_sql = sqlparse.format(
                str(statement),
                strip_comments=True,  # Keep comments if needed
                reindent=True,
                keyword_case='upper'
            ).strip()
            
            # Only add non-empty statements
            if formatted_sql and formatted_sql not in ['', ';']:
                sections.append(formatted_sql)
        
        return sections if sections else [sql_content.strip()]
    
    except Exception as e:
        print(f"Error parsing SQL: {str(e)}")
        # Return original content as fallback
        return [sql_content.strip()]

def get_statement_type(sql_statement):
    """
    Identify the type of SQL statement (SELECT, INSERT, CREATE, etc.)
    """
    try:
        parsed = sqlparse.parse(sql_statement)[0]
        stmt_type = parsed.get_type()
        return stmt_type
    except:
        return "UNKNOWN"

def process_query_files(source_path):
    """
    Main function to read files, split queries using sqlparse, and write to Unity Catalog table.
    """
    
    # Define the source table path
    
    query_output_schema = StructType([
    StructField("query_id",StringType(), True),        
    StructField("source_query", StringType(), True),
    StructField("total_commands", IntegerType(), True)
    ])
    
    command_output_schema = StructType([
    StructField("query_id", StringType(), True),
    StructField("command_number", IntegerType(), True),
    StructField("source_command", StringType(), True),
    StructField("command_type", StringType(), True)
    ])

    command_output_data=[]
    query_output_data=[]
    for filename in os.listdir(source_path):
    # Open the file and read its content
        try:
            with open(os.path.join(source_path, filename), 'r') as file:
                query_content = file.read()
                query_id=str(uuid.uuid4())

                # Split the query into sections using sqlparse
                sections = split_sql_with_sqlparse(query_content)

                command_count=len(sections)

                query_output_data.append({
                'query_id': query_id,
                'source_query': query_content,
                'total_commands': command_count
                })


                print(f"  - Split into {len(sections)} sections")
                
                # Create records for each section
                for section_idx, section in enumerate(sections):
                    statement_type = get_statement_type(section)
                    
                    command_output_data.append({
                        'query_id': query_id,
                        'command_number': section_idx + 1,
                        'source_command': section,
                        'command_type': statement_type
                    })
    
            
            # print(f"\nTotal sections created: {len(output_data)}")

        except Exception as e:
            print(f"✗ Error processing files: {str(e)}")
            import traceback
            traceback.print_exc()
            raise
            
    # Create DataFrame from output data
    query_output_df = spark.createDataFrame(query_output_data, schema=query_output_schema)
     # Define output table (same as source table - will overwrite)
    query_output_table = "learn_adb_fikrat.cc_bronze.source_target_queries"
    
    # Write to Unity Catalog table
    query_output_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(query_output_table)
    
       # Create DataFrame from output data
    command_output_df = spark.createDataFrame(command_output_data, schema=command_output_schema)
     # Define output table (same as source table - will overwrite)
    command_output_table = "learn_adb_fikrat.cc_bronze.source_target_commands"
    
    # Write to Unity Catalog table
    command_output_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(command_output_table)
 
     

In [0]:
process_query_files('/Volumes/learn_adb_fikrat/cc_bronze/source_queries')

In [0]:
%sql
select * from learn_adb_fikrat.cc_bronze.source_target_queries

In [0]:
%sql
select * from learn_adb_fikrat.cc_bronze.source_target_commands

In [0]:
%sql
create function learn_adb_fikrat.cc_bronze.get_source_commands ()
RETURNS TABLE (
  command_number INT,
  source_command STRING
)
RETURN (
  SELECT command_number, source_command from learn_adb_fikrat.cc_bronze.source_target_commands)


In [0]:
sql_command="""
WITH RecentOrders AS (
    SELECT 
        o.OrderID,
        o.CustomerID,
        o.OrderDate,
        o.TotalAmount
    FROM Orders o
    WHERE o.OrderDate >= DATEADD(MONTH, -6, GETDATE())  -- last 6 months
),
TopCustomers AS (
    SELECT 
        CustomerID,
        SUM(TotalAmount) AS TotalSpent
    FROM RecentOrders
    GROUP BY CustomerID
    HAVING SUM(TotalAmount) > 5000  -- high-value customers
),
ProductSales AS (
    SELECT 
        p.ProductID,
        p.ProductName,
        SUM(od.Quantity) AS TotalQty,
        SUM(od.Quantity * od.UnitPrice) AS Revenue
    FROM OrderDetails od
    INNER JOIN Products p ON od.ProductID = p.ProductID
    GROUP BY p.ProductID, p.ProductName
),
CustomerDetails AS (
    SELECT 
        c.CustomerID,
        c.CompanyName,
        c.City,
        c.Country
    FROM Customers c
    WHERE c.Country IN ('USA', 'Canada')
)
-- Main Query: Combine high-value customers with their recent orders and top products
SELECT 
    tc.CustomerID,
    cd.CompanyName,
    cd.City,
    cd.Country,
    ro.OrderID,
    ro.OrderDate,
    ro.TotalAmount,
    ps.ProductName,
    ps.Revenue
FROM TopCustomers tc
INNER JOIN CustomerDetails cd ON tc.CustomerID = cd.CustomerID
INNER JOIN RecentOrders ro ON tc.CustomerID = ro.CustomerID
LEFT JOIN ProductSales ps ON ps.Revenue > 10000  -- only top-selling products
WHERE ro.TotalAmount > 100
UNION
-- Include customers with no recent orders but high product purchases
SELECT 
    cd.CustomerID,
    cd.CompanyName,
    cd.City,
    cd.Country,
    NULL AS OrderID,
    NULL AS OrderDate,
    NULL AS TotalAmount,
    ps.ProductName,
    ps.Revenue
FROM CustomerDetails cd
INNER JOIN ProductSales ps ON ps.Revenue > 20000
ORDER BY Revenue DESC;
"""

In [0]:
sql_command="""
SELECT * FROM cc_bronze.source_queries where file_name='AdventureWorksDW2019';
UPDATE cc_bronze.source_queries SET query='test';
DELETE FROM cc_bronze.source_queries WHERE file_name='AdventureWorksDW2019';
"""

In [0]:
sections=split_sql_with_sqlparse(sql_command)

In [0]:
sections